# Unified Model Evaluation Notebook
Supports OpenAI, Claude, Gemini, and OpenRouter with proper tokenization

In [12]:
import os
import re
import json
import base64
import time
from datetime import datetime
import pandas as pd
from tqdm.notebook import tqdm
from dotenv import load_dotenv
from typing import Optional, Dict, Any, Tuple

# Load environment variables
load_dotenv()

# Import based on what's available
try:
    import tiktoken
    TIKTOKEN_AVAILABLE = True
except ImportError:
    TIKTOKEN_AVAILABLE = False
    print("Warning: tiktoken not available. Install with: pip install tiktoken")

try:
    from transformers import GPT2TokenizerFast
    TRANSFORMERS_AVAILABLE = True
except ImportError:
    TRANSFORMERS_AVAILABLE = False
    print("Warning: transformers not available. Install with: pip install transformers")

try:
    from openai import OpenAI
    OPENAI_AVAILABLE = True
except ImportError:
    OPENAI_AVAILABLE = False
    print("Warning: openai not available. Install with: pip install openai")

try:
    from anthropic import Anthropic
    ANTHROPIC_AVAILABLE = True
except ImportError:
    ANTHROPIC_AVAILABLE = False
    print("Warning: anthropic not available. Install with: pip install anthropic")

try:
    import google.generativeai as genai
    GOOGLE_AVAILABLE = True
except ImportError:
    GOOGLE_AVAILABLE = False
    print("Warning: google-generativeai not available. Install with: pip install google-generativeai")

In [ ]:
# Model configurations
MODEL_CONFIGS = {
    # ============= OpenAI Models (Direct API) =============
    
    "gpt-5.2-2025-12-11": {
        "provider": "openai",
        "context_limit": 400_000,
        "cost_per_input": 0.00000175,
        "cost_per_output": 0.000014,
        "supports_vision": True
    }
}

In [14]:
# Initialize tokenizers
class TokenizerManager:
    def __init__(self):
        self.tokenizers = {}
        
        # Initialize Claude tokenizer if available
        if TRANSFORMERS_AVAILABLE:
            try:
                self.tokenizers['claude'] = GPT2TokenizerFast.from_pretrained('Xenova/claude-tokenizer')
                print("Claude tokenizer loaded")
            except Exception as e:
                print(f"Could not load Claude tokenizer: {e}")
                self.tokenizers['claude'] = None
        
        # Initialize Gemini/general tokenizer
        if TRANSFORMERS_AVAILABLE:
            try:
                self.tokenizers['general'] = GPT2TokenizerFast.from_pretrained('gpt2')
                print("General tokenizer loaded")
            except Exception as e:
                print(f"Could not load general tokenizer: {e}")
                self.tokenizers['general'] = None
    
    def get_tokenizer(self, model_name: str):
        """Get appropriate tokenizer for model"""
        if model_name not in MODEL_CONFIGS:
            print(f"Warning: Unknown model {model_name}, using general tokenizer")
            provider = "general"
        else:
            provider = MODEL_CONFIGS[model_name]["provider"]
        
        # For OpenRouter models, determine tokenizer based on model name prefix
        if provider == "openrouter":
            if "openai" in model_name.lower() or "gpt" in model_name.lower():
                # OpenAI models via OpenRouter
                if TIKTOKEN_AVAILABLE:
                    try:
                        return tiktoken.get_encoding("cl100k_base")
                    except Exception:
                        return self.tokenizers.get('general')
                return self.tokenizers.get('general')
            elif "anthropic" in model_name.lower() or "claude" in model_name.lower():
                # Claude models via OpenRouter
                return self.tokenizers.get('claude') or self.tokenizers.get('general')
            else:
                # Other OpenRouter models (Gemini, Deepseek, etc.)
                return self.tokenizers.get('general')
        
        # For OpenAI models (direct API), use tiktoken
        if provider == "openai" and TIKTOKEN_AVAILABLE:
            try:
                return tiktoken.encoding_for_model(model_name)
            except KeyError:
                print(f"Model {model_name} not found in tiktoken, using cl100k_base")
                return tiktoken.get_encoding("cl100k_base")
        
        # For Claude models (direct API)
        if provider == "anthropic":
            return self.tokenizers.get('claude') or self.tokenizers.get('general')
        
        # For Google and others
        return self.tokenizers.get('general')
    
    def count_tokens(self, text: str, model_name: str) -> int:
        """Count tokens in text for given model"""
        tokenizer = self.get_tokenizer(model_name)
        if tokenizer is None:
            # Fallback to word count approximation
            return int(len(text.split()) * 1.3)
        
        if hasattr(tokenizer, 'encode'):
            return len(tokenizer.encode(text))
        else:
            return int(len(text.split()) * 1.3)
    
    def truncate_text(self, text: str, model_name: str, max_tokens: int) -> str:
        """Truncate text to max_tokens for given model"""
        tokenizer = self.get_tokenizer(model_name)
        if tokenizer is None:
            words = text.split()
            return ' '.join(words[-int(max_tokens/1.3):])
        
        try:
            tokens = tokenizer.encode(text)
            if len(tokens) <= max_tokens:
                return text
            
            # Keep the end of the text (most relevant for questions)
            truncated_tokens = tokens[-max_tokens:]
            
            if hasattr(tokenizer, 'decode'):
                return tokenizer.decode(truncated_tokens)
            else:
                words = text.split()
                return ' '.join(words[-int(max_tokens/1.3):])
        except Exception as e:
            print(f"Error truncating text: {e}")
            words = text.split()
            return ' '.join(words[-int(max_tokens/1.3):])

tokenizer_manager = TokenizerManager()


Claude tokenizer loaded
General tokenizer loaded


In [15]:
# Unified API caller
class UnifiedAPIClient:
    def __init__(self):
        self.openai_client = None
        self.anthropic_client = None
        self.google_client = None
        self.openrouter_api_key = None
        self.openrouter_base_url = None
        self.last_provider = None  # Track last provider to avoid repetitive messages
        
        # Initialize OpenRouter configuration (we'll use requests directly)
        if USE_OPENROUTER and OPENROUTER_API_KEY:
            self.openrouter_api_key = OPENROUTER_API_KEY
            self.openrouter_base_url = OPENROUTER_BASE_URL
            print("OpenRouter configured (using requests library)")
        
        # Initialize provider-specific clients
        if OPENAI_AVAILABLE and os.environ.get('OPENAI_API_KEY'):
            self.openai_client = OpenAI()
            print("OpenAI client initialized")
        
        if ANTHROPIC_AVAILABLE and os.environ.get('ANTHROPIC_API_KEY'):
            self.anthropic_client = Anthropic()
            print("Anthropic client initialized")
        
        if GOOGLE_AVAILABLE and os.environ.get('GOOGLE_API_KEY'):
            genai.configure(api_key=os.environ.get('GOOGLE_API_KEY'))
            print("Google AI configured")
    
    def _uses_max_completion_tokens(self, model: str) -> bool:
        """
        Detect if model uses max_completion_tokens instead of max_tokens.
        
        GPT-5+ models and o1/o3 models use max_completion_tokens.
        Older models (GPT-4, GPT-4o) use max_tokens.
        """
        model_lower = model.lower()
        
        # Check for GPT-5+ models (including versions like gpt-5.2, gpt-5-mini, gpt-5-nano)
        if "gpt-5" in model_lower or "gpt5" in model_lower:
            return True
        
        # Check for o1 and o3 models
        if model.startswith(("o1-", "o3-")):
            return True
        
        return False
    
    def call_model(self, model: str, text: str, image_path: Optional[str] = None, 
                   max_tokens: int = 4096, temperature: float = 0, seed: int = 42) -> Dict[str, Any]:
        """Unified model calling interface"""
        
        if model not in MODEL_CONFIGS:
            raise ValueError(f"Unknown model: {model}")
        
        config = MODEL_CONFIGS[model]
        provider = config["provider"]
        
        # Route based on provider
        if provider == "openrouter":
            # OpenRouter models always use requests directly (for provider forcing)
            if not self.openrouter_api_key:
                raise RuntimeError("OpenRouter not configured. Set USE_OPENROUTER=true and OPENROUTER_API_KEY in .env")
            return self._call_openrouter(model, text, image_path, max_tokens, temperature, config, seed=seed)
        elif USE_OPENROUTER and self.openrouter_api_key:
            # If USE_OPENROUTER is set, route all models through OpenRouter
            return self._call_openrouter(model, text, image_path, max_tokens, temperature, config, seed=seed)
        elif provider == "openai":
            return self._call_openai(model, text, image_path, max_tokens, temperature, config, seed=seed)
        elif provider == "anthropic":
            return self._call_anthropic(model, text, image_path, max_tokens, temperature, config)
        elif provider == "google":
            return self._call_google(model, text, image_path, max_tokens, temperature, config)
        else:
            raise ValueError(f"Unknown provider: {provider}")
    
    def _call_openrouter(self, model, text, image_path, max_tokens, temperature, config, seed: int = 42):
        """Call OpenRouter API using requests (supports provider forcing)"""
        import requests
        
        messages = [
            {
                "role": "system",
                "content": "You are a data analyst. I will give you a background introduction and data analysis question. You must answer the question."
            }
        ]
        
        user_content = []
        
        if image_path and config["supports_vision"]:
            base64_image = self._encode_image(image_path)
            user_content.append({
                "type": "text",
                "text": text
            })
            user_content.append({
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{base64_image}"
                }
            })
        else:
            user_content.append({
                "type": "text",
                "text": text
            })
        
        messages.append({
            "role": "user",
            "content": user_content
        })
        
        # Determine which token parameter to use based on model
        if self._uses_max_completion_tokens(model):
            token_param = "max_completion_tokens"
        else:
            token_param = "max_tokens"
        
        # Build payload with dynamic token parameter
        payload = {
            "model": model,
            "messages": messages,
            "temperature": temperature,
            token_param: max_tokens,  # Dynamic parameter name
        }
        
        # Add seed for deterministic results
        if seed is not None:
            payload["seed"] = seed
        
        # Add provider forcing (prevents fallback to other providers)
        # Extract provider from model name (e.g., "openai/gpt-5.2" -> "OpenAI")
        # Note: Some models don't work well with provider forcing, so we skip them
        provider_map = {
            "openai": "OpenAI",
            "anthropic": "Anthropic",
            "google": "Google",
            # "deepseek": Skip - provider forcing causes 404 errors
            "z-ai": "Zhipu",
            "xiaomi": "Xiaomi",
            "x-ai": "xAI"
        }
        
        model_prefix = model.split("/")[0].lower()
        provider_name = provider_map.get(model_prefix)
        
        # Determine current provider status (for tracking)
        if provider_name:
            current_provider = f"locked:{provider_name}"
        elif "deepseek" in model_prefix:
            current_provider = "auto:deepseek"
        else:
            current_provider = "auto:unknown"
        
        # Only print if provider changed
        if current_provider != self.last_provider:
            if provider_name:
                # Force specific provider, disable fallbacks
                print(f"  🔒 Provider locked to: {provider_name}")
            else:
                # No provider forcing - let OpenRouter route automatically
                if "deepseek" in model_prefix:
                    print(f"  🔓 Provider forcing disabled (using OpenRouter auto-routing)")
            
            self.last_provider = current_provider
        
        # Set provider in payload if forcing
        if provider_name:
            payload["provider"] = {
                "order": [provider_name],
                "allow_fallbacks": False
            }
        
        # Make request
        headers = {
            "Authorization": f"Bearer {self.openrouter_api_key}",
            "Content-Type": "application/json",
        }
        
        try:
            response = requests.post(
                f"{self.openrouter_base_url}/chat/completions",
                headers=headers,
                json=payload,
                timeout=120,
            )
            response.raise_for_status()
            result = response.json()
            
            return {
                "response": result["choices"][0]["message"]["content"],
                "input_tokens": result["usage"]["prompt_tokens"],
                "output_tokens": result["usage"]["completion_tokens"],
                "model": result.get("model", model)
            }
        except requests.exceptions.HTTPError as e:
            # Enhanced error logging for OpenRouter
            error_msg = str(e)
            try:
                error_details = e.response.json()
                error_msg = f"{error_msg}\nDetails: {json.dumps(error_details, indent=2)}"
            except:
                pass
            print(f"  ❌ OpenRouter Error: {error_msg}")
            raise Exception(error_msg)
    
    def _call_openai(self, model, text, image_path, max_tokens, temperature, config, seed: int = 42):
        """Call OpenAI API"""
        if not self.openai_client:
            raise RuntimeError("OpenAI client not initialized")
        
        messages = [
            {
                "role": "system",
                "content": "You are a data analyst. I will give you a background introduction and data analysis question. You must answer the question."
            }
        ]
        
        user_content = []
        
        if image_path and config["supports_vision"]:
            base64_image = self._encode_image(image_path)
            user_content.append({
                "type": "text",
                "text": text
            })
            user_content.append({
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{base64_image}"
                }
            })
        else:
            user_content.append({
                "type": "text",
                "text": text
            })
        
        messages.append({
            "role": "user",
            "content": user_content
        })
        
        # Determine which token parameter to use based on model
        if self._uses_max_completion_tokens(model):
            token_param = "max_completion_tokens"
        else:
            token_param = "max_tokens"
        
        # Build request params with dynamic token parameter
        request_params = {
            "model": model,
            "messages": messages,
            "temperature": temperature,
            token_param: max_tokens,  # Dynamic parameter name
            "top_p": 1
        }
        
        if seed is not None:
            request_params["seed"] = seed
        
        response = self.openai_client.chat.completions.create(**request_params)
        
        return {
            "response": response.choices[0].message.content,
            "input_tokens": response.usage.prompt_tokens,
            "output_tokens": response.usage.completion_tokens,
            "model": response.model
        }
    
    def _call_anthropic(self, model, text, image_path, max_tokens, temperature, config):
        """Call Anthropic API"""
        if not self.anthropic_client:
            raise RuntimeError("Anthropic client not initialized")
        
        messages = []
        user_content = []
        
        if image_path and config["supports_vision"]:
            base64_image = self._encode_image(image_path)
            user_content.append({
                "type": "image",
                "source": {
                    "type": "base64",
                    "media_type": "image/jpeg",
                    "data": base64_image
                }
            })
        
        user_content.append({
            "type": "text",
            "text": text
        })
        
        messages.append({
            "role": "user",
            "content": user_content
        })
        
        response = self.anthropic_client.messages.create(
            model=model,
            max_tokens=max_tokens,
            temperature=temperature,
            system="You are a data analyst. I will give you a background introduction and data analysis question. You must answer the question.",
            messages=messages
        )
        
        return {
            "response": response.content[0].text,
            "input_tokens": response.usage.input_tokens,
            "output_tokens": response.usage.output_tokens,
            "model": response.model
        }
    
    def _call_google(self, model, text, image_path, max_tokens, temperature, config):
        """Call Google Gemini API"""
        if not GOOGLE_AVAILABLE:
            raise RuntimeError("Google AI not available")
        
        client = genai.GenerativeModel(model)
        
        system_msg = "You are a data analyst. I will give you a background introduction and data analysis question. You must answer the question."
        
        inputs = [system_msg + "\n" + text]
        
        if image_path and config["supports_vision"]:
            from PIL import Image
            image = Image.open(image_path)
            inputs.append(image)
        
        response = client.generate_content(
            inputs,
            generation_config=genai.types.GenerationConfig(
                temperature=temperature,
                max_output_tokens=max_tokens
            )
        )
        
        # Calculate cost based on token usage
        input_tokens = response.usage_metadata.prompt_token_count
        output_tokens = response.usage_metadata.candidates_token_count
        
        return {
            "response": response.text,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "model": model
        }
    
    def _encode_image(self, image_path: str) -> str:
        """Encode image to base64"""
        with open(image_path, "rb") as image_file:
            return base64.b64encode(image_file.read()).decode('utf-8')

api_client = UnifiedAPIClient()

OpenAI client initialized


In [16]:
# Utility functions
def find_jpg_files(directory):
    """Find image files in directory"""
    jpg_files = [file for file in os.listdir(directory) 
                 if file.lower().endswith(('.jpg', '.png', '.jpeg'))]
    return jpg_files if jpg_files else None

def find_excel_files(directory):
    """Find Excel files in directory"""
    excel_files = [file for file in os.listdir(directory) 
                   if (file.lower().endswith(('.xlsx', '.xlsb', '.xlsm')) 
                       and not "answer" in file.lower()
                       and not file.startswith('~$'))]  # Exclude Excel temp files
    return excel_files if excel_files else None

def read_excel(file_path):
    """Read Excel file and return all sheets"""
    try:
        if file_path.lower().endswith('.xlsb'):
            xls = pd.ExcelFile(file_path, engine='pyxlsb')
        else:
            xls = pd.ExcelFile(file_path, engine='openpyxl')
        
        sheets = {}
        for sheet_name in xls.sheet_names:
            sheets[sheet_name] = xls.parse(sheet_name)
        return sheets
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return {}

def dataframe_to_text(df):
    """Convert DataFrame to text"""
    return df.to_string(index=False)

def combine_sheets_text(sheets):
    """Combine all sheets into text"""
    combined_text = ""
    for sheet_name, df in sheets.items():
        sheet_text = dataframe_to_text(df)
        combined_text += f"Sheet name: {sheet_name}\n{sheet_text}\n\n"
    return combined_text

def read_txt(path):
    """Read text file"""
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

def calculate_cost(input_tokens: int, output_tokens: int, model: str) -> float:
    """Calculate cost for a model call"""
    if model not in MODEL_CONFIGS:
        return 0.0
    
    config = MODEL_CONFIGS[model]
    cost = (input_tokens * config["cost_per_input"] + 
            output_tokens * config["cost_per_output"])
    return cost

def retry_with_backoff(func, max_retries=5, initial_delay=1):
    """Retry function with exponential backoff for transient errors"""
    for attempt in range(max_retries):
        try:
            return func()
        except Exception as e:
            error_msg = str(e).lower()
            error_type = type(e).__name__
            
            # Transient errors that should be retried
            is_rate_limit = "rate" in error_msg or "quota" in error_msg or "429" in error_msg
            is_timeout = "timeout" in error_msg or "timed out" in error_msg
            is_network = "connection" in error_msg or "network" in error_msg or "connectionerror" in error_type.lower()
            is_server_error = "503" in error_msg or "502" in error_msg or "500" in error_msg or "bad gateway" in error_msg
            
            # These are retriable transient errors
            is_retriable = is_rate_limit or is_timeout or is_network or is_server_error
            
            if is_retriable and attempt < max_retries - 1:
                delay = initial_delay * (2 ** attempt)
                error_category = (
                    "Rate limit" if is_rate_limit else
                    "Timeout" if is_timeout else
                    "Network error" if is_network else
                    "Server error"
                )
                print(f"  {error_category} detected. Retrying in {delay}s... (attempt {attempt+1}/{max_retries})")
                time.sleep(delay)
                continue
            
            # For non-retriable errors or final attempt, raise immediately
            raise e
    
    raise Exception(f"Failed after {max_retries} retries")

## Run Evaluation

Configure your model below and run the evaluation

### 🔑 Using OpenRouter for All Models

If you want to use OpenRouter as a unified gateway for **all models** (not just OpenRouter-specific ones), set in your `.env`:

```env
USE_OPENROUTER=true
OPENROUTER_API_KEY=your_openrouter_key
```

**Benefits:**
- ✅ Single API key for all providers
- ✅ Provider forcing (no fallbacks)
- ✅ Seed support for reproducibility
- ✅ Consistent interface

When `USE_OPENROUTER=true`, all models will route through OpenRouter, even OpenAI/Claude/Gemini models.

In [17]:
# ========== CONFIGURATION ==========
# Change these parameters for your evaluation

# Model to evaluate (choose from MODEL_CONFIGS keys)
# MODEL = "deepseek/deepseek-r1-0528:free"  # Change to any model in MODEL_CONFIGS
# MODEL = "google/gemini-3-pro-preview"
# MODEL = "anthropic/claude-opus-4.5"
# MODEL = "openai/gpt-4o-2024-11-20"
# MODEL = "gpt-4o-2024-11-20"
# MODEL = "deepseek/deepseek-r1-0528:free"
MODEL = "gpt-5.2-2025-12-11"

# Data path and Data JSON path
# DATA_PATH = "data_subset_olive"
DATA_TAG = "data_subset_olive"  # Change to your data tag

DATA_PATH = f"./{DATA_TAG}/"
DATA_JSON_FILE = f"./{DATA_TAG}.json"

# Tokens reserved for generation (context_limit - tokens4generation = input limit)
TOKENS_FOR_GENERATION = 6000

# Max tokens for model output
MAX_OUTPUT_TOKENS = 4096

# Temperature (0 for deterministic, higher for creative)
TEMPERATURE = 0

# Seed for reproducibility (used with temperature=0 for deterministic results)
SEED = 42

# Auto-resume from most recent run (set to False to be prompted)
AUTO_RESUME = False

# ===================================

if MODEL not in MODEL_CONFIGS:
    raise ValueError(f"Model {MODEL} not found in MODEL_CONFIGS. Available models: {list(MODEL_CONFIGS.keys())}")

model_config = MODEL_CONFIGS[MODEL]
context_limit = model_config["context_limit"]
max_input_tokens = context_limit - TOKENS_FOR_GENERATION

print(f"Evaluating model: {MODEL}")
print(f"Provider: {model_config['provider']}")
print(f"Context limit: {context_limit:,}")
print(f"Max input tokens: {max_input_tokens:,}")
print(f"Tokens for generation: {TOKENS_FOR_GENERATION:,}")
print(f"Supports vision: {model_config['supports_vision']}")
print(f"Temperature: {TEMPERATURE}")
print(f"Seed: {SEED}")

Evaluating model: gpt-5.2-2025-12-11
Provider: openai
Context limit: 400,000
Max input tokens: 394,000
Tokens for generation: 6,000
Supports vision: True
Temperature: 0
Seed: 42


In [18]:
# Load samples
samples = []
data_file = DATA_JSON_FILE  # Change this to your data file

with open(data_file, "r") as f:
    for line in f:
        samples.append(eval(line.strip()))

print(f"Loaded {len(samples)} samples")

Loaded 22 samples


In [19]:
# ========== FOLDER MANAGEMENT & RESUME ==========
# This cell handles folder selection: resume from existing run or create new

import glob

def find_existing_runs(base_path: str, model_name: str):
    """Find all existing evaluation runs for a model"""
    # Look for timestamped subdirs inside model_name folder
    model_folder = os.path.join(base_path, model_name)
    
    if not os.path.exists(model_folder):
        return []
    
    # Get all subdirectories (timestamps)
    subdirs = [d for d in os.listdir(model_folder) 
               if os.path.isdir(os.path.join(model_folder, d))]
    
    runs = []
    for subdir in subdirs:
        folder_path = os.path.join(model_folder, subdir)
        json_files = glob.glob(os.path.join(folder_path, "*.json"))
        
        # Filter out results.json and results_process.json
        sample_files = [f for f in json_files 
                       if os.path.basename(f) not in ['results.json', 'results_process.json']]
        
        # Count total samples and questions
        total_samples = len(sample_files)
        total_questions = 0
        total_errors = 0
        
        for json_file in sample_files:
            with open(json_file, 'r') as f:
                lines = f.readlines()
                for line in lines:
                    try:
                        entry = json.loads(line.strip())
                        if "error" in entry:
                            total_errors += 1
                        else:
                            total_questions += 1
                    except:
                        pass
        
        # Get folder creation time
        ctime = os.path.getctime(folder_path)
        mtime = os.path.getmtime(folder_path)
        runs.append({
            "folder": folder_path,
            "folder_name": subdir,  # Just the timestamp part
            "samples": total_samples,
            "questions": total_questions,
            "errors": total_errors,
            "created": datetime.fromtimestamp(ctime),
            "modified": datetime.fromtimestamp(mtime),
        })
    
    # Sort by modification time (most recent first)
    runs.sort(key=lambda x: x["modified"], reverse=True)
    return runs


def display_runs(runs):
    """Display existing runs in a formatted way"""
    if not runs:
        print("No existing runs found.")
        return
    
    print("\n" + "="*80)
    print("EXISTING EVALUATION RUNS")
    print("="*80)
    
    for i, run in enumerate(runs, 1):
        print(f"\n{i}. {run['folder_name']}")
        print(f"   Created: {run['created'].strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Modified: {run['modified'].strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Samples: {run['samples']} | Questions: {run['questions']} | Errors: {run['errors']}")
        
        # Show completion status
        if run['errors'] > 0:
            print(f"   ⚠️  Status: {run['errors']} questions have errors (need re-run)")
        else:
            print(f"   ✅ Status: All questions successful")


def cleanup_errors(folder_path: str, dry_run: bool = True) -> int:
    """
    Remove error entries from JSON files
    
    Args:
        folder_path: Path to evaluation folder
        dry_run: If True, only show what would be removed (don't actually remove)
    
    Returns:
        Number of error entries removed
    """
    json_files = glob.glob(os.path.join(folder_path, "*.json"))
    
    # Filter out results.json and results_process.json
    sample_files = [f for f in json_files 
                   if os.path.basename(f) not in ['results.json', 'results_process.json']]
    
    total_removed = 0
    
    print(f"\n{'DRY RUN: ' if dry_run else ''}Cleaning error entries from {folder_path}")
    print("="*80)
    
    for json_file in sample_files:
        with open(json_file, 'r') as f:
            lines = f.readlines()
        
        valid_lines = []
        removed_count = 0
        
        for line in lines:
            try:
                entry = json.loads(line.strip())
                if "error" in entry:
                    removed_count += 1
                    if dry_run:
                        print(f"   Would remove error from {os.path.basename(json_file)}: {entry.get('error', '')[:60]}")
                else:
                    valid_lines.append(line)
            except:
                pass
        
        if removed_count > 0:
            total_removed += removed_count
            if not dry_run:
                # Actually remove
                with open(json_file, 'w') as f:
                    f.writelines(valid_lines)
                print(f"   ✅ Removed {removed_count} error(s) from {os.path.basename(json_file)}")
    
    if dry_run and total_removed > 0:
        print(f"\n💡 Run with dry_run=False to actually remove {total_removed} error entries")
    elif total_removed == 0:
        print("   ✅ No errors found")
    
    return total_removed


# ========== FOLDER SELECTION LOGIC ==========

sanitized_model_name = MODEL.replace('/', '_').replace(':', '_')
base_path = "./save_process"

# Find existing runs for this model
runs = find_existing_runs(base_path, sanitized_model_name)

if not runs:
    # No existing runs, create new one
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    save_path = os.path.join(base_path, sanitized_model_name, timestamp)
    os.makedirs(save_path, exist_ok=True)
    print(f"\n✨ No existing runs found. Creating new folder:")
    print(f"   {save_path}")
else:
    # Display existing runs
    display_runs(runs)
    
    if AUTO_RESUME:
        # Auto-resume from most recent
        most_recent = runs[0]
        save_path = most_recent['folder']
        print(f"\n🔄 AUTO-RESUMING from most recent run:")
        print(f"   {save_path}")
    else:
        # Ask user
        print("\n" + "="*80)
        print("OPTIONS:")
        print("="*80)
        for i, run in enumerate(runs, 1):
            status = "⚠️  Has errors" if run['errors'] > 0 else "✅ Complete"
            print(f"{i}. Resume from: {run['folder_name']} ({status})")
        print(f"{len(runs) + 1}. Create NEW run (with timestamp)")
        print("="*80)
        
        while True:
            try:
                choice = input(f"\nEnter choice (1-{len(runs) + 1}): ").strip()
                choice_num = int(choice)
                
                if 1 <= choice_num <= len(runs):
                    # Resume from existing
                    selected = runs[choice_num - 1]
                    save_path = selected['folder']
                    print(f"\n🔄 RESUMING from:")
                    print(f"   {save_path}")
                    
                    # Ask about cleaning errors
                    if selected['errors'] > 0:
                        print(f"\n⚠️  This run has {selected['errors']} error entries.")
                        cleanup_choice = input("Clean errors before resuming? (y/n): ").strip().lower()
                        if cleanup_choice == 'y':
                            # Show what would be removed
                            cleanup_errors(save_path, dry_run=True)
                            confirm = input("\nProceed with cleanup? (y/n): ").strip().lower()
                            if confirm == 'y':
                                cleanup_errors(save_path, dry_run=False)
                                print("✅ Errors cleaned")
                    break
                elif choice_num == len(runs) + 1:
                    # Create new
                    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                    save_path = os.path.join(base_path, sanitized_model_name, timestamp)
                    os.makedirs(save_path, exist_ok=True)
                    print(f"\n✨ Creating NEW run:")
                    print(f"   {save_path}")
                    break
                else:
                    print(f"Invalid choice. Please enter 1-{len(runs) + 1}")
            except ValueError:
                print(f"Invalid input. Please enter a number (1-{len(runs) + 1})")
            except KeyboardInterrupt:
                print("\n\n❌ Cancelled by user")
                raise

print(f"\n{'='*80}")
print(f"📁 Evaluation folder ready: {save_path}")
print(f"{'='*80}")


EXISTING EVALUATION RUNS

1. 20260203_183355
   Created: 2026-02-03 18:33:55
   Modified: 2026-02-03 18:34:39
   Samples: 3 | Questions: 0 | Errors: 30
   ⚠️  Status: 30 questions have errors (need re-run)

OPTIONS:
1. Resume from: 20260203_183355 (⚠️  Has errors)
2. Create NEW run (with timestamp)

✨ Creating NEW run:
   ./save_process\gpt-5.2-2025-12-11\20260203_192001

📁 Evaluation folder ready: ./save_process\gpt-5.2-2025-12-11\20260203_192001


In [20]:
# Main evaluation loop
total_cost_this_run = 0  # Cost for NEW questions in this run
total_cost_timestamp = 0  # Total cost across ALL runs for this timestamp
total_samples = 0
total_questions = 0

for sample_idx in tqdm(range(len(samples)), desc="Processing samples"):
    sample = samples[sample_idx]
    
    # Skip if no questions
    if len(sample["questions"]) == 0:
        continue
    
    # Determine which questions are already successfully processed
    output_file = os.path.join(save_path, f"{sample['id']}.json")
    processed_indices = set()
    existing_cost = 0
    existing_time = 0
    
    if os.path.exists(output_file):
        with open(output_file, "r") as f:
            for line in f:
                try:
                    entry = json.loads(line.strip())
                    # Only count as processed if NO error field
                    if "error" not in entry:
                        processed_indices.add(entry["question_idx"])
                        # Accumulate existing costs from completed questions
                        existing_cost += entry.get("cost", 0)
                        existing_time += entry.get("time", 0)
                except:
                    pass
        
        # Add existing costs to timestamp total (regardless of whether we skip)
        total_cost_timestamp += existing_cost
        
        if len(processed_indices) >= len(sample["questions"]):
            print(f"Skipping {sample['id']} - all questions completed ({len(processed_indices)}/{len(sample['questions'])})")
            continue
        else:
            print(f"\nResuming {sample['id']} - {len(processed_indices)}/{len(sample['questions'])} questions already successful")
            print(f"Will process {len(sample['questions']) - len(processed_indices)} remaining/errored questions")
            if existing_cost > 0:
                print(f"💰 Existing costs from completed questions: ${existing_cost:.6f}")
    
    # Find image if exists
    image_path = None
    image_files = find_jpg_files(os.path.join(DATA_PATH, sample["id"]))
    if image_files:
        image_path = os.path.join(DATA_PATH, sample["id"], image_files[0])
    
    # Read Excel files
    excel_content = ""
    excel_files = find_excel_files(os.path.join(DATA_PATH, sample["id"]))
    if excel_files:
        for excel_file in excel_files:
            excel_path = os.path.join(DATA_PATH, sample["id"], excel_file)
            sheets = read_excel(excel_path)
            combined_text = combine_sheets_text(sheets)
            excel_content += f"The excel file {excel_file} is: {combined_text}\n"
    
    # Read introduction
    introduction = read_txt(os.path.join(DATA_PATH, sample["id"], "introduction.txt"))
    
    # Read questions
    questions = []
    for question_name in sample["questions"]:
        question_text = read_txt(os.path.join(DATA_PATH, sample["id"], f"{question_name}.txt"))
        questions.append(question_text)
    
    # Build base context
    base_text = ""
    if excel_content:
        base_text += f"The workbook is detailed as follows. {excel_content}\n"
    base_text += f"The introduction is detailed as follows.\n{introduction}\n"
    
    # Track costs for this sample (NEW questions only)
    sample_new_cost = 0
    sample_new_questions = 0
    
    # Open file for appending
    with open(output_file, "a") as save_file:
        print(f"\nProcessing sample {sample['id']} ({sample_idx + 1}/{len(samples)})...")
        
        # Process all questions, but skip already-processed ones
        for question_idx, question in enumerate(questions):
            # Skip if already successfully processed
            if question_idx in processed_indices:
                print(f"  Question {question_idx + 1}/{len(questions)}: ✓ Already completed")
                continue
            
            # Build full prompt
            prompt = base_text + f"The questions are detailed as follows.\n{question}"
            
            # print("The PROMPT is:", prompt)  #DEBUG PROMPT  

            # Truncate if needed
            truncated_prompt = tokenizer_manager.truncate_text(prompt, MODEL, max_input_tokens)
            
            # Display question number
            print(f"  Question {question_idx + 1}/{len(questions)}: ", end="", flush=True)
            
            # Call model with retry logic
            def make_call():
                start_time = time.time()
                result = api_client.call_model(
                    model=MODEL,
                    text=truncated_prompt,
                    image_path=image_path,
                    max_tokens=MAX_OUTPUT_TOKENS,
                    temperature=TEMPERATURE,
                    seed=SEED  # Pass seed for reproducibility
                )
                result["time"] = time.time() - start_time
                return result
            
            try:
                result = retry_with_backoff(make_call, max_retries=5, initial_delay=5)
                
                # Calculate cost
                cost = calculate_cost(result["input_tokens"], result["output_tokens"], MODEL)
                
                # Update totals
                total_cost_this_run += cost
                total_cost_timestamp += cost
                total_questions += 1
                sample_new_cost += cost
                sample_new_questions += 1
                
                # Print cost information (per-question + cumulative for timestamp)
                print(f"In: {result['input_tokens']:,} tok | Out: {result['output_tokens']:,} tok | Cost: ${cost:.6f} | Total: ${total_cost_timestamp:.6f}")
                
                # Build answer object
                answer = {
                    "id": sample["id"],
                    "question_idx": question_idx,
                    "model": result["model"],
                    "input_tokens": result["input_tokens"],
                    "output_tokens": result["output_tokens"],
                    "cost": cost,
                    "time": result["time"],
                    "response": result["response"]
                }
                
                # Save immediately
                json.dump(answer, save_file)
                save_file.write("\n")
                save_file.flush()
                
            except Exception as e:
                print(f"❌ ERROR: {e}")
                # Save error info
                error_answer = {
                    "id": sample["id"],
                    "question_idx": question_idx,
                    "error": str(e),
                    "model": MODEL
                }
                json.dump(error_answer, save_file)
                save_file.write("\n")
                save_file.flush()
                continue
    
    total_samples += 1
    
    # Calculate total sample cost (existing + new)
    total_sample_cost = existing_cost + sample_new_cost
    
    print(f"\n{'='*80}")
    print(f"✅ Completed {sample['id']}: {len(questions)} questions")
    if existing_cost > 0:
        print(f"💰 Sample cost: ${total_sample_cost:.6f} (${existing_cost:.6f} existing + ${sample_new_cost:.6f} new)")
    else:
        print(f"💰 Sample cost: ${sample_new_cost:.6f}")
    print(f"💰 Timestamp total so far: ${total_cost_timestamp:.6f}")
    print(f"{'='*80}\n")

print(f"\n{'='*80}")
print(f"🎉 Evaluation complete!")
print(f"Model: {MODEL}")
print(f"Samples processed: {total_samples}")
print(f"Questions processed (new): {total_questions}")
print(f"")
print(f"💰 Total Cost for this run: ${total_cost_this_run:.6f}")
print(f"💰 Total Cost for current timestamp (all runs): ${total_cost_timestamp:.6f}")
if total_questions > 0:
    print(f"")
    print(f"Average Cost per Question (new): ${total_cost_this_run/total_questions:.6f}")
print(f"")
print(f"Results saved to: {save_path}")
print(f"{'='*80}")

Processing samples:   0%|          | 0/22 [00:00<?, ?it/s]


Processing sample 00000001 (1/22)...
Model gpt-5.2-2025-12-11 not found in tiktoken, using cl100k_base
  Question 1/13: In: 18,989 tok | Out: 127 tok | Cost: $0.035009 | Total: $0.035009
Model gpt-5.2-2025-12-11 not found in tiktoken, using cl100k_base
  Question 2/13: In: 19,008 tok | Out: 454 tok | Cost: $0.039620 | Total: $0.074629
Model gpt-5.2-2025-12-11 not found in tiktoken, using cl100k_base
  Question 3/13: In: 19,004 tok | Out: 136 tok | Cost: $0.035161 | Total: $0.109790
Model gpt-5.2-2025-12-11 not found in tiktoken, using cl100k_base
  Question 4/13: In: 19,040 tok | Out: 161 tok | Cost: $0.035574 | Total: $0.145364
Model gpt-5.2-2025-12-11 not found in tiktoken, using cl100k_base
  Question 5/13: In: 19,034 tok | Out: 593 tok | Cost: $0.041611 | Total: $0.186975
Model gpt-5.2-2025-12-11 not found in tiktoken, using cl100k_base
  Question 6/13: In: 19,042 tok | Out: 383 tok | Cost: $0.038685 | Total: $0.225661
Model gpt-5.2-2025-12-11 not found in tiktoken, using cl100k_b

c:\Users\ustra\miniconda3\envs\dsbench\lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)



Processing sample 00000033 (20/22)...
Model gpt-5.2-2025-12-11 not found in tiktoken, using cl100k_base
  Question 1/8: In: 75,760 tok | Out: 66 tok | Cost: $0.133504 | Total: $13.926013
Model gpt-5.2-2025-12-11 not found in tiktoken, using cl100k_base
  Question 2/8: In: 75,768 tok | Out: 133 tok | Cost: $0.134456 | Total: $14.060469
Model gpt-5.2-2025-12-11 not found in tiktoken, using cl100k_base
  Question 3/8: In: 75,763 tok | Out: 70 tok | Cost: $0.133565 | Total: $14.194035
Model gpt-5.2-2025-12-11 not found in tiktoken, using cl100k_base
  Question 4/8: In: 75,768 tok | Out: 54 tok | Cost: $0.133350 | Total: $14.327385
Model gpt-5.2-2025-12-11 not found in tiktoken, using cl100k_base
  Question 5/8: In: 75,756 tok | Out: 34 tok | Cost: $0.133049 | Total: $14.460434
Model gpt-5.2-2025-12-11 not found in tiktoken, using cl100k_base
  Question 6/8: In: 75,768 tok | Out: 79 tok | Cost: $0.133700 | Total: $14.594134
Model gpt-5.2-2025-12-11 not found in tiktoken, using cl100k_base


c:\Users\ustra\miniconda3\envs\dsbench\lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


In: 2,914 tok | Out: 346 tok | Cost: $0.009944 | Total: $21.576413
Model gpt-5.2-2025-12-11 not found in tiktoken, using cl100k_base
  Question 2/5: In: 2,906 tok | Out: 204 tok | Cost: $0.007942 | Total: $21.584355
Model gpt-5.2-2025-12-11 not found in tiktoken, using cl100k_base
  Question 3/5: In: 2,919 tok | Out: 348 tok | Cost: $0.009980 | Total: $21.594335
Model gpt-5.2-2025-12-11 not found in tiktoken, using cl100k_base
  Question 4/5: In: 2,913 tok | Out: 174 tok | Cost: $0.007534 | Total: $21.601869
Model gpt-5.2-2025-12-11 not found in tiktoken, using cl100k_base
  Question 5/5: In: 2,905 tok | Out: 1,117 tok | Cost: $0.020722 | Total: $21.622590

✅ Completed 00000038: 5 questions
💰 Sample cost: $0.056121
💰 Timestamp total so far: $21.622590


🎉 Evaluation complete!
Model: gpt-5.2-2025-12-11
Samples processed: 22
Questions processed (new): 236

💰 Total Cost for this run: $21.622590
💰 Total Cost for current timestamp (all runs): $21.622590

Average Cost per Question (new): $0.